# Track C: NeMo Data Designer — 한국어 수학 CoT 데이터셋 생성

vLLM으로 로컬 배포한 Nemotron 3를 활용하여, 한국 초·중등 수학 문제와 단계별 Chain-of-Thought(CoT) 풀이를 생성하는 합성 데이터 파이프라인.

**아키텍처:**
```
Nemotron 3 (vLLM) <-- OpenAI API --> NeMo Data Designer --> 한국어 수학 CoT 데이터셋
```

## Step 0: 환경 설정 & vLLM 배포

In [ ]:
# Step 0.1: 패키지 설치
!uv sync

### 0.2a vLLM 패키지 설치 (GPU 서버에서 실행)

아래 셀은 GPU 서버 환경에서만 실행하세요. 로컬 Mac에서는 건너뛰세요.

In [ ]:
%%script bash
# Step 0.2a: vLLM 패키지 설치 (GPU 서버에서만 실행)
set -euo pipefail
pip install -U vllm==0.17.1 torch==2.10.0 flashinfer-python==0.6.4 \
  flashinfer-cubin==0.6.4 'nvidia-cutlass-dsl>=4.4.0.dev1' \
  --extra-index-url https://download.pytorch.org/whl/cu128

### 0.2 vLLM 배포

**별도 터미널에서** 아래 스크립트를 실행하세요. 이 셀은 실행하지 마세요.

| 모델 | 명령어 | 필요 GPU |
|------|--------|----------|
| Super BF16 | `bash scripts/launch_nemotron_super.sh bf16` | 4x H100 80GB |
| Super FP8 | `bash scripts/launch_nemotron_super.sh fp8` | 2x H100 80GB |
| Super NVFP4 | `bash scripts/launch_nemotron_super.sh nvfp4` | 1x B200 |
| Nano BF16 | `bash scripts/launch_nemotron_nano.sh bf16` | 1x H100/A100 80GB |
| Nano FP8 | `bash scripts/launch_nemotron_nano.sh fp8` | 1x H100 80GB |
| Nano NVFP4 | `bash scripts/launch_nemotron_nano.sh nvfp4` | 1x B200 |

준비되면 로그에 `Uvicorn running on http://0.0.0.0:5000` 이 보입니다.

In [ ]:
# Step 0.3: vLLM 서버 연결 설정
import os
from dotenv import load_dotenv

load_dotenv()

VLLM_BASE_URL = os.environ.get("NVIDIA_BASE_URL", "http://localhost:5000/v1")
VLLM_MODEL_NAME = "nvidia/nemotron-3-nano-30b-a3b"
# VLLM_MODEL_NAME = "nvidia/nemotron-3-super-120b-a12b"  # 고성능

print(f"Base URL: {VLLM_BASE_URL}")
print(f"Model: {VLLM_MODEL_NAME}")

In [ ]:
# Data Designer 환경변수 설정
os.environ["NVIDIA_BASE_URL"] = VLLM_BASE_URL
os.environ["NVIDIA_API_KEY"] = os.environ.get("NVIDIA_API_KEY", "not-used")

print(f"Configured: {VLLM_BASE_URL}")

In [ ]:
from data_designer.interface import DataDesigner
from data_designer.config import (
    CategorySamplerParams,
    ChatCompletionInferenceParams,
    DataDesignerConfigBuilder,
    ExpressionColumnConfig,
    LLMJudgeColumnConfig,
    LLMStructuredColumnConfig,
    LLMTextColumnConfig,
    ModelConfig,
    ModelProvider,
    SamplerColumnConfig,
    SamplerType,
    Score,
)
from pydantic import BaseModel, Field
from typing import List

print("Import 완료!")

## Step 1: 데이터 스키마 정의

| 컬럼 | 타입 | 설명 |
|------|------|------|
| grade | Sampler | 대상 학년 (초3–중3) |
| topic | Sampler | 수학 주제 영역 |
| difficulty | Sampler | 쉬움/보통/어려움 |
| problem | LLM Text | 한국어 수학 문제 |
| solution | LLM Text | 단계별 CoT 풀이 |
| answer_metadata | LLM Structured | 정답 + 사용 개념 |
| quality_score | LLM Judge | 품질 평가 점수 |
| chat_formatted | Expression | SFT 대화 형식 |

In [ ]:
# 1.1 모델 설정

vllm_provider = ModelProvider(
    name="vllm-local",
    endpoint=VLLM_BASE_URL,
    provider_type="openai",
    api_key=os.environ.get("NVIDIA_API_KEY", "not-used"),
)

problem_model = ModelConfig(
    alias="problem-generator",
    model=VLLM_MODEL_NAME,
    provider="vllm-local",
    inference_parameters=ChatCompletionInferenceParams(
        temperature=0.9, top_p=0.95, max_tokens=4096,
    ),
)

solution_model = ModelConfig(
    alias="solution-generator",
    model=VLLM_MODEL_NAME,
    provider="vllm-local",
    inference_parameters=ChatCompletionInferenceParams(
        temperature=0.3, top_p=0.9, max_tokens=2048,
    ),
)

structured_model = ModelConfig(
    alias="extractor",
    model=VLLM_MODEL_NAME,
    provider="vllm-local",
    inference_parameters=ChatCompletionInferenceParams(
        temperature=0.1, max_tokens=512,
    ),
)

judge_model = ModelConfig(
    alias="quality-judge",
    model=VLLM_MODEL_NAME,
    provider="vllm-local",
    inference_parameters=ChatCompletionInferenceParams(
        temperature=0.1, max_tokens=1024,
    ),
)

print(f"모델 설정 완료 ({VLLM_BASE_URL})")

In [ ]:
# 1.2 샘플러 컬럼 — 학년, 주제, 난이도 분포 제어

grade_column = SamplerColumnConfig(
    name="grade",
    sampler_type=SamplerType.CATEGORY,
    params=CategorySamplerParams(
        values=[
            "초등학교 3학년", "초등학교 4학년", "초등학교 5학년", "초등학교 6학년",
            "중학교 1학년", "중학교 2학년", "중학교 3학년",
        ],
        weights=[0.10, 0.12, 0.15, 0.15, 0.18, 0.15, 0.15],
    ),
)

topic_column = SamplerColumnConfig(
    name="topic",
    sampler_type=SamplerType.CATEGORY,
    params=CategorySamplerParams(
        values=[
            "사칙연산 (Arithmetic)",
            "분수와 소수 (Fractions & Decimals)",
            "도형과 측정 (Geometry & Measurement)",
            "비와 비율 (Ratios & Proportions)",
            "방정식 (Equations)",
            "함수와 그래프 (Functions & Graphs)",
            "확률과 통계 (Probability & Statistics)",
            "규칙과 패턴 (Patterns & Sequences)",
        ],
        weights=[0.15, 0.15, 0.12, 0.12, 0.13, 0.10, 0.10, 0.13],
    ),
)

difficulty_column = SamplerColumnConfig(
    name="difficulty",
    sampler_type=SamplerType.CATEGORY,
    params=CategorySamplerParams(
        values=["쉬움 (Easy)", "보통 (Medium)", "어려움 (Hard)"],
        weights=[0.25, 0.50, 0.25],
    ),
)

print("Sampler columns defined: grade, topic, difficulty")

In [ ]:
# 1.3 LLM 텍스트 컬럼 — 문제 & 풀이 생성

PROBLEM_PROMPT = """\
You are an expert Korean math education specialist.

Create a math problem in Korean for the following specifications:
- Grade level: {{ grade }}
- Topic: {{ topic }}
- Difficulty: {{ difficulty }}

Requirements:
- The problem MUST be written entirely in Korean.
- Use a real-world context that Korean students can relate to \
(e.g., school life, shopping, cooking, sports, travel in Korea).
- The problem should be solvable with knowledge appropriate for the given grade level.
- Include specific numbers and clearly state what needs to be found.
- Do NOT include the solution or answer — only the problem statement.

Output ONLY the problem text in Korean, nothing else.

Format:
## 문제
[problem text]
"""

problem_column = LLMTextColumnConfig(
    name="problem",
    prompt=PROBLEM_PROMPT,
    model_alias="problem-generator",
)

SOLUTION_PROMPT = """\
You are an expert Korean math tutor who explains solutions step by step.

Solve the following math problem with a detailed Chain-of-Thought explanation in Korean.

Problem ({{ grade }}, {{ topic }}, {{ difficulty }}):
{{ problem }}

Requirements:
- Write the ENTIRE solution in Korean.
- Break down the solution into clear, numbered steps (풀이 과정).
- Explain the reasoning behind each step so a {{ grade }} student can understand.
- Use mathematical notation where appropriate (e.g., +, -, ×, ÷, =).
- End with the final answer clearly marked as: **정답: [answer]**

Format:
## 풀이 과정

1단계: [first step explanation]
2단계: [second step explanation]
...

**정답: [final answer]**
"""

solution_column = LLMTextColumnConfig(
    name="solution",
    prompt=SOLUTION_PROMPT,
    model_alias="solution-generator",
)

print("LLM text columns defined: problem, solution")

In [ ]:
# 1.4 구조화 출력 컬럼 — 정답 메타데이터 추출

class MathAnswerMetadata(BaseModel):
    """Structured metadata extracted from a math problem and its solution."""
    final_answer: str = Field(description="The final numerical or expression answer")
    num_steps: int = Field(description="Number of reasoning steps in the solution")
    concepts_used: List[str] = Field(description="List of math concepts used (in Korean)")
    requires_diagram: bool = Field(description="Whether the problem would benefit from a diagram")

METADATA_PROMPT = """\
Analyze the following Korean math problem and its solution.
Extract the requested metadata accurately.

Problem: {{ problem }}
Solution: {{ solution }}

Extract:
1. The final answer (just the value/expression)
2. The number of reasoning steps
3. List of mathematical concepts used (in Korean, e.g., ["덧셈", "곱셈", "비율"])
4. Whether a diagram would help understand the problem
"""

metadata_column = LLMStructuredColumnConfig(
    name="answer_metadata",
    prompt=METADATA_PROMPT,
    output_format=MathAnswerMetadata,
    model_alias="extractor",
)

print("Structured column defined: answer_metadata")
print(f"Schema: {MathAnswerMetadata.model_json_schema()}")

In [ ]:
# 1.5 LLM Judge 컬럼 — 품질 평가 (1-5점 루브릭)

problem_quality_score = Score(
    name="problem_quality",
    description="Overall quality of the math problem: clarity, fit for the stated grade, and use of realistic context.",
    options={
        1: "Unclear or off-grade; not solvable as stated or ambiguous.",
        2: "Mostly understandable but context feels unnatural or difficulty is inappropriate.",
        3: "Clear, grade-appropriate, and uses reasonable real-life context.",
        4: "Strong: clear, pedagogical, and context fits Korean students' everyday situations.",
        5: "Excellent: creative, high educational value, ideal difficulty and context.",
    },
)

solution_correctness_score = Score(
    name="solution_correctness",
    description="Mathematical correctness and logical completeness: each step and the final answer.",
    options={
        1: "Serious math errors; wrong calculations or invalid reasoning.",
        2: "Right direction but notable calculation mistakes or gaps in logic.",
        3: "Mathematically sound and logically valid; final answer is correct.",
        4: "Accurate with clear step-by-step explanation; easy for a student to follow.",
        5: "Flawless: accurate, excellent stepwise explanation, strong pedagogical insight.",
    },
)

cot_quality_score = Score(
    name="cot_quality",
    description="Quality of the chain-of-thought: step structure, justification of each step, and readability for the target grade.",
    options={
        1: "Little explanation or answer-only; not real CoT.",
        2: "Some steps but explanations are thin or not at the right level.",
        3: "Adequate step-by-step explanation; a student can follow along.",
        4: "Detailed and educational; rationale for each step is explicit.",
        5: "Outstanding: intuitive, highly educational, and supports mathematical thinking.",
    },
)

JUDGE_PROMPT = """\
You are an expert evaluator for Korean K-12 mathematics.

Evaluate the following **Korean** word problem and **Korean** step-by-step solution \
using the scoring rubrics (configured separately).

**Language policy**
- The problem and solution text below are in Korean; judge them on that content.
- For any rationale, brief comments, or free-form reasoning you output as part of this \
judgment, write in **Korean** (한국어).

**Grade:** {{ grade }}
**Topic:** {{ topic }}
**Difficulty:** {{ difficulty }}

**Problem (Korean):**
{{ problem }}

**Solution (Korean):**
{{ solution }}

Assign scores according to the rubric.
"""

quality_judge_column = LLMJudgeColumnConfig(
    name="quality_score",
    prompt=JUDGE_PROMPT,
    model_alias="quality-judge",
    scores=[problem_quality_score, solution_correctness_score, cot_quality_score],
)

print("LLM Judge column defined: quality_score")
print("Rubrics: problem_quality (1-5), solution_correctness (1-5), cot_quality (1-5)")

In [ ]:
# 1.6 표현식 컬럼 — SFT 대화 형식

SYSTEM_MESSAGE = "당신은 한국 수학 교육 전문가입니다. 학생의 수준에 맞춰 친절하고 단계적으로 수학 문제를 풀어주세요."

chat_column = ExpressionColumnConfig(
    name="chat_formatted",
    expr=(
        '[{"role": "system", "content": "' + SYSTEM_MESSAGE + '"},'
        ' {"role": "user", "content": "{{ problem }}"},'
        ' {"role": "assistant", "content": "{{ solution }}"}]'
    ),
)

print("Expression column defined: chat_formatted")

## Step 2: 파이프라인 구성 & 미리보기

In [ ]:
config_builder = DataDesignerConfigBuilder(
    model_configs=[problem_model, solution_model, structured_model, judge_model],
)

config_builder.add_column(grade_column)
config_builder.add_column(topic_column)
config_builder.add_column(difficulty_column)
config_builder.add_column(problem_column)
config_builder.add_column(solution_column)
config_builder.add_column(metadata_column)
config_builder.add_column(quality_judge_column)
config_builder.add_column(chat_column)

print("파이프라인 컬럼 등록 완료.")

In [ ]:
# 미리보기: 5개 샘플 생성
data_designer = DataDesigner(model_providers=[vllm_provider])

data_designer.validate(config_builder)
print("설정 검증 완료.")

preview = data_designer.preview(config_builder=config_builder, num_records=5)
print(f"{len(preview.dataset)}개의 미리보기 레코드 생성 완료!")

In [ ]:
preview.display_sample_record(index=0)

In [ ]:
df_preview = preview.dataset
df_preview[["grade", "topic", "difficulty", "problem", "quality_score", "answer_metadata"]].head()

In [ ]:
# 전체 예제 상세 출력
row = df_preview.iloc[0]

print("=" * 60)
print(f"학년:   {row['grade']}")
print(f"주제:   {row['topic']}")
print(f"난이도: {row['difficulty']}")
print("=" * 60)
print(f"\n문제:\n{row['problem']}")
print(f"\n풀이:\n{row['solution']}")
print(f"\n메타데이터:\n{row['answer_metadata']}")
print(f"\n품질 평가:\n{row['quality_score']}")
print("=" * 60)

## Step 3: 대규모 생성

In [ ]:
NUM_RECORDS = 30  # 프로덕션: 500-1000+

creation_results = data_designer.create(
    config_builder=config_builder,
    num_records=NUM_RECORDS,
    dataset_name="korean-math-cot",
)
print("\n생성 완료!")

In [ ]:
df = creation_results.load_dataset()

print(f"총 생성 레코드 수: {len(df)}")
print(f"\n컬럼: {list(df.columns)}")
print(f"\n학년 분포:\n{df['grade'].value_counts()}")
print(f"\n주제 분포:\n{df['topic'].value_counts()}")
print(f"\n난이도 분포:\n{df['difficulty'].value_counts()}")

## Step 4: 품질 분석 & LLM Judge 기반 필터링

In [ ]:
import json
import pandas as pd

def parse_metadata(meta_str):
    if isinstance(meta_str, dict):
        return meta_str
    try:
        return json.loads(meta_str)
    except (json.JSONDecodeError, TypeError):
        return None

def parse_judge_scores(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    if isinstance(val, dict):
        return val
    if isinstance(val, str):
        try:
            return json.loads(val)
        except (json.JSONDecodeError, TypeError):
            return None
    return None

def extract_criterion_score(parsed, name):
    if not isinstance(parsed, dict):
        return None
    v = parsed.get(name)
    if v is None:
        return None
    if isinstance(v, dict):
        return v.get("score")
    return v

df["parsed_meta"] = df["answer_metadata"].apply(parse_metadata)
df["parsed_scores"] = df["quality_score"].apply(parse_judge_scores)

for key, col in [
    ("problem_quality", "score_problem_quality"),
    ("solution_correctness", "score_solution_correctness"),
    ("cot_quality", "score_cot_quality"),
]:
    df[col] = df["parsed_scores"].apply(lambda x, k=key: extract_criterion_score(x, k))

for col in ["score_problem_quality", "score_solution_correctness", "score_cot_quality"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

score_cols = ["score_problem_quality", "score_solution_correctness", "score_cot_quality"]
df["avg_quality_score"] = df[score_cols].mean(axis=1)

df["solution_length"] = df["solution"].str.len()
df["problem_length"] = df["problem"].str.len()
df["has_steps"] = df["solution"].str.contains("단계", na=False)
df["has_answer"] = df["solution"].str.contains("정답", na=False)

print("=== 품질 지표 ===")
print(f"  평균 풀이 길이: {df['solution_length'].mean():.0f}자")
print(f"  평균 문제 길이: {df['problem_length'].mean():.0f}자")
print(f"  단계 표시 포함: {df['has_steps'].sum()}/{len(df)}")
print(f"  정답 표시 포함: {df['has_answer'].sum()}/{len(df)}")
print()
print("=== LLM Judge 평가 점수 (1-5점) ===")
for c in score_cols:
    print(f"  {c}: {df[c].mean():.2f}")

In [ ]:
# LLM Judge 점수 기반 고품질 필터링
JUDGE_THRESHOLD_MIN = 3
JUDGE_THRESHOLD_AVG = 3.5

df_filtered = df[
    (df["has_steps"]) &
    (df["has_answer"]) &
    (df["solution_length"] > 200) &
    (df["parsed_meta"].notna()) &
    (df["score_problem_quality"] >= JUDGE_THRESHOLD_MIN) &
    (df["score_solution_correctness"] >= JUDGE_THRESHOLD_MIN) &
    (df["score_cot_quality"] >= JUDGE_THRESHOLD_MIN) &
    (df["avg_quality_score"] >= JUDGE_THRESHOLD_AVG)
].copy()

print(f"필터링 결과: {len(df_filtered)}/{len(df)}개 ({len(df_filtered)/len(df)*100:.1f}% 유지)")
print(f"필터링 후 평균 품질 점수: {df_filtered['avg_quality_score'].mean():.2f}")

## Step 4.5: NeMo Curator — 후처리 (중복 제거 & 품질 필터링)

Data Designer로 생성한 데이터셋에 NeMo Curator를 적용하여 중복을 제거하고 최종 품질을 보장합니다.

```
Data Designer (생성) → LLM Judge (필터링) → NeMo Curator (후처리) → 최종 데이터셋
```

| 기능 | 설명 |
|------|------|
| 정확 중복 제거 | 해시 기반으로 완전히 동일한 문서 제거 |
| 퍼지 중복 제거 | MinHash LSH로 의미적으로 유사한 문서 제거 |
| 커스텀 품질 필터 | 한국어 비율, 최소 길이 등 휴리스틱 필터링 |

In [ ]:
# NeMo Curator 설치 (GPU 가속 버전)
# !pip install nemo-curator            # CPU 버전
# !pip install "nemo-curator[cuda12x]" # GPU 가속 버전

import nemo_curator as nc
from nemo_curator.datasets import DocumentDataset
from nemo_curator.modules import ExactDuplicates

# Data Designer 출력을 NeMo Curator DocumentDataset으로 변환
dataset = DocumentDataset.from_pandas(df_filtered, text_field="problem")

# 정확 중복 제거: 동일한 문제를 제거
exact_dedup = ExactDuplicates(id_field="id", text_field="problem")
deduplicated = exact_dedup(dataset)
df_dedup = deduplicated.to_pandas()
print(f"정확 중복 제거: {len(df_filtered)} → {len(df_dedup)}개 ({len(df_filtered) - len(df_dedup)}개 제거)")

In [ ]:
# 퍼지 중복 제거 (MinHash LSH) — 유사한 문제 탐지
from nemo_curator.modules import FuzzyDuplicates, FuzzyDuplicatesConfig

fuzzy_config = FuzzyDuplicatesConfig(
    seed=42,
    char_ngrams=5,
    num_buckets=20,
    hashes_per_bucket=13,
    use_64_bit_hash=False,
    buckets_per_shuffle=5,
    false_positive_check=True,
    num_anchors=2,
    jaccard_threshold=0.8,  # 80% 이상 유사하면 중복으로 판정
)

fuzzy_dedup = FuzzyDuplicates(
    config=fuzzy_config,
    id_field="id",
    text_field="problem",
)
result = fuzzy_dedup(dataset)
df_fuzzy = result.to_pandas()
print(f"퍼지 중복 제거 후: {len(df_fuzzy)}개")

In [ ]:
# 커스텀 품질 필터 — 한국어 수학 CoT 전용
from nemo_curator.filters import DocumentFilter

class KoreanMathQualityFilter(DocumentFilter):
    """한국어 수학 CoT 데이터셋 전용 품질 필터."""

    def __init__(self, min_problem_len=50, min_solution_len=200):
        super().__init__()
        self.min_problem_len = min_problem_len
        self.min_solution_len = min_solution_len

    def score_document(self, text: str) -> bool:
        if len(text) < self.min_problem_len:
            return False
        # 한국어 포함 여부 체크
        korean_chars = sum(1 for c in text if '\uac00' <= c <= '\ud7a3')
        if korean_chars / max(len(text), 1) < 0.3:
            return False
        return True

    def keep_document(self, score) -> bool:
        return score

quality_filter = nc.ScoreFilter(
    KoreanMathQualityFilter(min_problem_len=50),
    text_field="problem",
)
filtered_dataset = quality_filter(dataset)
df_curator_final = filtered_dataset.to_pandas()
print(f"커스텀 품질 필터 후: {len(df_curator_final)}개")

## Step 5: 데이터셋 내보내기

In [ ]:
import json
from pathlib import Path

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

sft_records = []
for _, row in df_filtered.iterrows():
    record = {
        "messages": [
            {"role": "system", "content": "당신은 한국 수학 교육 전문가입니다. 학생의 수준에 맞춰 친절하고 단계적으로 수학 문제를 풀어주세요."},
            {"role": "user", "content": row["problem"]},
            {"role": "assistant", "content": row["solution"]},
        ],
        "metadata": {
            "grade": row["grade"],
            "topic": row["topic"],
            "difficulty": row["difficulty"],
            "quality_scores": {
                "problem_quality": row.get("score_problem_quality"),
                "solution_correctness": row.get("score_solution_correctness"),
                "cot_quality": row.get("score_cot_quality"),
                "avg_score": row.get("avg_quality_score"),
            },
        },
    }
    sft_records.append(record)

sft_path = output_dir / "korean_math_cot_sft.jsonl"
with open(sft_path, "w", encoding="utf-8") as f:
    for record in sft_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"{len(sft_records)}개 레코드를 {sft_path}에 내보냈습니다")

parquet_path = output_dir / "korean_math_cot_full.parquet"
df_filtered.to_parquet(parquet_path, index=False)
print(f"전체 데이터셋을 {parquet_path}에 저장했습니다")

## Step 6 (선택): HuggingFace Hub 업로드

In [ ]:
# Uncomment after setting HF_TOKEN

# import os
# os.environ["HF_TOKEN"] = "hf_..."
#
# url = creation_results.push_to_hub(
#     repo_id="your-username/korean-math-cot-dataset",
#     private=False,
# )
# print(f"Dataset pushed to: {url}")

## Step 7 (보너스): vLLM 직접 API 테스트

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=VLLM_BASE_URL, api_key="not-used")

response = client.chat.completions.create(
    model=VLLM_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful math tutor. Answer in Korean."},
        {"role": "user", "content": "3 + 5 x 2 = ? 풀이 과정을 알려주세요."},
    ],
    temperature=0.3,
    max_tokens=512,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print("vLLM Direct Test Response:")
print(response.choices[0].message.content)